In [1]:
import duckdb

In [3]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [18]:
# Initialize database path
db_file = '../olist_medallion_architecture.db'

# Connect to database file
conn = duckdb.connect(db_file, read_only = True)

## Data Dictionary (Views & Tables)

In [3]:
# Verify views exist in data dictionary
conn.query("""
    SELECT view_name
    FROM duckdb_views()
    WHERE NOT internal
        AND view_name LIKE 'silver%' OR view_name LIKE 'gold%'
    ORDER BY
        view_name;
""")

view_nameVARCHAR
silver_customers
silver_geolocation
silver_order_items
silver_orders
silver_payments
silver_products
silver_sellers


In [11]:
# Verify tables exist in data dictionary
conn.query("""
    SELECT table_name
    FROM duckdb_tables();
""")

table_nameVARCHAR
gold_customers
gold_order_payments_products
gold_sellers
silver_reviews


## Payments Table 

In [10]:
# Evaluate specific order_id
conn.query("""
    SELECT *
    FROM silver_payments
    WHERE order_id = '524b32821380cdcaa2b6d3110eebcd7e'
    ORDER BY 
        payment_sequence;
""")

order_idVARCHAR,payment_sequenceBIGINT,typeVARCHAR,installmentsBIGINT,valueDOUBLE
524b32821380cdcaa2b6d3110eebcd7e,1,credit_card,2,31.23
524b32821380cdcaa2b6d3110eebcd7e,2,voucher,1,91.13
524b32821380cdcaa2b6d3110eebcd7e,3,voucher,1,9.11
524b32821380cdcaa2b6d3110eebcd7e,4,voucher,1,9.11
524b32821380cdcaa2b6d3110eebcd7e,5,voucher,1,9.11
524b32821380cdcaa2b6d3110eebcd7e,6,voucher,1,9.11


In [13]:
conn.query("""
    WITH tmp AS (
        SELECT
            type,
            COUNT(*) AS ct
        FROM silver_payments
        GROUP BY 
            type
    )
    SELECT *
    FROM tmp
    WHERE ct > 1
    ORDER BY 
        ct DESC;
""")

typeVARCHAR,ctBIGINT
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [ ]:
# Pre-aggregate payments table
conn.query("""
    WITH tmp AS (
        SELECT
            order_id,
            ARG_MAX(type, value) AS largest_single_payment_type,
            ARG_MIN(type, payment_sequence) AS initial_payment_type,
            LIST(DISTINCT type ORDER BY type) AS payment_methods_used,
            -- Option 1: (Mixed payment type categorization)
            CASE 
                WHEN COUNT(DISTINCT type) = 1 THEN MAX(type)
                WHEN COUNT(DISTINCT type) > 1 
                    AND LIST_CONTAINS(LIST(DISTINCT type), 'voucher') 
                    AND LIST_CONTAINS(LIST(DISTINCT type), 'credit_card') 
                    THEN 'mixed_voucher_card'
                ELSE 'other_split_payment'
            END AS payment_category,
            -- Option 2: (Boolean flag for quick filtering)
            COUNT(DISTINCT type) > 1 AS is_split_payment,
            MAX(payment_sequence) AS total_transactions,
            MAX(installments) AS total_installments,
            ROUND(SUM(value), 2) AS total_value
        FROM silver_payments
        GROUP BY 
            order_id
    )
    SELECT *
    FROM tmp
    WHERE initial_payment_type = 'boleto'
    --WHERE order_id = '524b32821380cdcaa2b6d3110eebcd7e';
""")

order_idVARCHAR,largest_single_payment_typeVARCHAR,initial_payment_typeVARCHAR,payment_methods_usedVARCHAR[],payment_categoryVARCHAR,is_split_paymentBOOLEAN,total_transactionsBIGINT,total_installmentsBIGINT,total_valueDOUBLE
5b7c6dbea3e21dc6314451ed73907c0c,boleto,boleto,['boleto'],boleto,False,1,1,48.27
0913c46b1cafecb52e5a9a55b6c0e0fc,boleto,boleto,['boleto'],boleto,False,1,1,147.0
940d83f3286808e4f8b1a7ac975b2aa9,boleto,boleto,['boleto'],boleto,False,1,1,76.02
a38d122ad3e07ca8d13e4a42059b98d2,boleto,boleto,['boleto'],boleto,False,1,1,161.62
c4b2b908aab6585b6a3147ac72952242,boleto,boleto,['boleto'],boleto,False,1,1,85.81
2df92600294b6acc114b86b79dfff0f3,boleto,boleto,['boleto'],boleto,False,1,1,44.23
58ecdd5dee96d19e30c67a29cae63e4d,boleto,boleto,['boleto'],boleto,False,1,1,105.28
c401e0f907a3a0912a86473918b3aaa0,boleto,boleto,['boleto'],boleto,False,1,1,139.28
ee748540786ca7a510e0889be7ef88ca,boleto,boleto,['boleto'],boleto,False,1,1,33.99
d0e368ca78daa02e421325f3d1b8541a,boleto,boleto,['boleto'],boleto,False,1,1,67.36


## Reviews Table
* Need to handle repeat reviews based on _order_id_

In [25]:
conn.query("""
    SELECT
        order_id,
        COUNT(*) AS ct
    FROM silver_reviews
    GROUP BY
        order_id
    HAVING ct > 1
    ORDER BY 
        ct DESC;
""")

order_idVARCHAR,ctBIGINT
03c939fd7fd3b38f8485a0f95798f1f6,3
c88b1d1b157a9999ce368f218a407141,3
df56136b8031ecd28e200bb18e6ddb2e,3
1041d437bb0a4bcfeb3a107426f2d5f5,2
824fa8179b9911d58831e0b09eedde13,2
4f68ed0836136db8064bf34ebb1965e2,2
3d905f2c25bbcb26fc068e0ca62a4315,2
3e27524d3fd47ae21dd03342be78a94f,2
f6089c47bab18faae3d90c64cfb08481,2
d61b915b69851aec8a8865f36cfd793e,2


In [38]:
# Pre-aggregate rating by order_id
conn.query("""
    SELECT
        order_id,
        DATE_TRUNC('month', creation_timestamp)::DATE AS creation_month,
        COUNT(order_id) AS review_count,
        ROUND(AVG(rating), 1) AS avg_rating
    FROM silver_reviews 
    GROUP BY
        order_id,
        DATE_TRUNC('month', creation_timestamp)
    HAVING review_count > 1;
""")

order_idVARCHAR,creation_monthDATE,review_countBIGINT,avg_ratingDOUBLE
029863af4b968de1e5d6a82782e662f5,2017-07-01,2,4.5
3ab974ac744bd19a111d5e19f0a90649,2017-10-01,2,5.0
c88b1d1b157a9999ce368f218a407141,2017-07-01,3,4.3
bf593abe2928b23b60cee1ff3471a4dd,2017-06-01,2,3.0
d78c16f09da03410837a551a09a4bf64,2017-07-01,2,3.5
e88f2d374afb179c85305c2e855f9d04,2017-09-01,2,3.0
6525d9ab0bff6133c63e9c0cfa55bfbd,2017-09-01,2,3.0
b8fdf0a70040d5461adca8f95ba4171a,2018-01-01,2,2.0
76de574e14e58baded25a12dce8075c1,2017-06-01,2,5.0
d898dd2a514f62601dc99b7858023038,2017-11-01,2,5.0


## Order Items Table

In [5]:
# Evaluate specific order_id OR product_id
conn.query("""
    SELECT * 
    FROM silver_order_items
    WHERE 1=1
        AND order_id = 'ca3625898fbd48669d50701aba51cd5f'
        --AND product_id = '1065e0ebef073787a7bf691924c60eeb';
""")

order_idVARCHAR,product_idVARCHAR,item_sequenceBIGINT,shipping_limit_timestampTIMESTAMP,item_priceDOUBLE,shipping_costDOUBLE
ca3625898fbd48669d50701aba51cd5f,1065e0ebef073787a7bf691924c60eeb,1,2018-08-16 02:25:07,309.0,1.84
ca3625898fbd48669d50701aba51cd5f,0cf2faf9749f53924cea652a09d8e327,2,2018-08-16 02:25:07,33.9,1.84
ca3625898fbd48669d50701aba51cd5f,0de59eddc63167215c972b0d785ffa7b,3,2018-08-16 02:25:07,159.0,3.67
ca3625898fbd48669d50701aba51cd5f,0de59eddc63167215c972b0d785ffa7b,4,2018-08-16 02:25:07,159.0,3.67
ca3625898fbd48669d50701aba51cd5f,5dae498eff2d80057f56122235a36aff,5,2018-08-17 02:25:07,95.9,0.15
ca3625898fbd48669d50701aba51cd5f,4a5c3967bfd3629fe07ef4d0cc8c3818,6,2018-08-16 02:25:07,109.9,0.15
ca3625898fbd48669d50701aba51cd5f,678c229b41c0e497d35a25a8be1cc631,7,2018-08-17 02:25:07,95.9,0.15
ca3625898fbd48669d50701aba51cd5f,21b524c4c060169fa75ccf08c7da4627,8,2018-08-16 02:25:07,63.7,0.15
ca3625898fbd48669d50701aba51cd5f,309dd69eb83cea38c51709d62befe1a4,9,2018-08-16 02:25:07,56.0,3.68
ca3625898fbd48669d50701aba51cd5f,309dd69eb83cea38c51709d62befe1a4,10,2018-08-16 02:25:07,56.0,3.68


In [ ]:
# Pre-aggregated order_items table (consider removing shipping_limit_timestamp)
conn.query("""
    SELECT
        order_id,
        MIN(shipping_limit_timestamp) AS min_shipping_limit_timestamp,
        MAX(shipping_limit_timestamp) AS max_shipping_limit_timestamp,

        COUNT(item_sequence) AS total_product_count,
        COUNT(DISTINCT product_id) AS unique_product_count,
        COUNT(DISTINCT seller_id) AS unique_seller_count,
        COALESCE(SUM(item_price), 0) AS total_item_price,
        COALESCE(SUM(shipping_cost), 0) AS total_shipping_cost,
        COALESCE(SUM(item_price + shipping_cost), 0) AS total_value
    FROM silver_order_items
    WHERE order_id = 'ca3625898fbd48669d50701aba51cd5f'
    GROUP BY 
        order_id
    ORDER BY 
        total_product_count DESC,
        order_id
""")

order_idVARCHAR,min_shipping_limit_timestampTIMESTAMP,max_shipping_limit_timestampTIMESTAMP,total_product_countBIGINT,unique_product_countBIGINT,unique_seller_countBIGINT,total_item_priceDOUBLE,total_shipping_costDOUBLE,total_valueDOUBLE
ca3625898fbd48669d50701aba51cd5f,2018-08-16 02:25:07,2018-08-17 02:25:07,10,8,2,1138.3,18.98,1157.28


## Orders Table

In [11]:
# Evaluate specific order_id OR customer_id
conn.query("""
    SELECT *
    FROM silver_orders
    WHERE 1=1
        --AND order_id = 'e481f51cbdc54678b7cc49136f2d6af7'
        AND customer_id = 'f88197465ea7920adcdbec7375364d82';
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,delivered_carrier_timestampTIMESTAMP,delivered_customer_timestampTIMESTAMP,estimated_delivery_timestampTIMESTAMP
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00


### Notes:
* 97% of _status_ values are 'delivered'
    * 1.1% are 'shipped', evaluate timestamps for these records
* Count the number of _order_id_ with the same _repeat_customer_id_
* 0.6% of _purchase_timestamp_ values are the same
* Compare _purchase_timestamp_ and _approved_timestamp_ (distribution of approval times and latency evaluation)
* Both _delivered_carrier_timestamp_ and _delivered_customer_timestamp_ have missing rows, what _status_ do these records have?
* Compare _delivered_customer_timestamp_ and _estimated_delivery_timestamp_ (join in _order_items.shipping_limit_timestamp_ and compare to both)
    * _estimated_delivery_timestamp_ only has 459 distinct values... whats going on here?

> Distribution of _status_ when package not yet delivered to customer (_delivery_customer_timestamp_ IS NULL)
* Status of [approved, invoiced, processing, created] are in queue
* Status of [unavailable] indicates product was not available in inventory
* Status of [canceled] indicates customer requested cancelation 

In [22]:
conn.query("""
    WITH tmp AS (
        SELECT
            order_id,
            customer_id,
            repeat_customer_id,
            status,
            purchase_timestamp,
            approved_timestamp,
            delivered_carrier_timestamp,
            delivered_customer_timestamp,
            estimated_delivery_timestamp
        FROM silver_orders
        WHERE 1=1
            --status = 'shipped'
            AND delivered_customer_timestamp IS NULL
    )
    SELECT
        status,
        COUNT(*) AS ct
    FROM tmp
    GROUP BY 
        status
    ORDER BY 
        ct DESC; 
""")

statusVARCHAR,ctBIGINT
shipped,1107
canceled,619
unavailable,609
invoiced,314
processing,301
delivered,8
created,5
approved,2


> Distribution of _status_ when order was not delivered to carrier (_delivered_carrier_timestamp_ IS NULL)

In [ ]:
conn.query("""
    WITH tmp AS (
        SELECT
            order_id,
            customer_id,
            repeat_customer_id,
            status,
            purchase_timestamp,
            approved_timestamp,
            delivered_carrier_timestamp,
            delivered_customer_timestamp,
            estimated_delivery_timestamp
        FROM silver_orders
        WHERE delivered_carrier_timestamp IS NULL
    )
    SELECT
        status,
        COUNT(*) AS ct
    FROM tmp
    GROUP BY 
        status
    ORDER BY 
        ct DESC; 
""")

statusVARCHAR,ctBIGINT
unavailable,609
canceled,550
invoiced,314
processing,301
created,5
delivered,2
approved,2


> One record shows order was delivered to customer, but never to carrier?

In [29]:
conn.query("""
    SELECT
        order_id,
        customer_id,
        repeat_customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        delivered_carrier_timestamp,
        delivered_customer_timestamp,
        estimated_delivery_timestamp
    FROM silver_orders
    WHERE delivered_carrier_timestamp IS NULL
        AND delivered_customer_timestamp IS NOT NULL
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,delivered_carrier_timestampTIMESTAMP,delivered_customer_timestampTIMESTAMP,estimated_delivery_timestampTIMESTAMP
2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,a2ac81ecc3704410ae240e74d4f0af40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NULL,2017-11-20 19:44:47,2017-11-14 00:00:00


> One record has NULL timestamp for both carrier and customer, but status is delivered...

In [30]:
conn.query("""
    SELECT
        order_id,
        customer_id,
        repeat_customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        delivered_carrier_timestamp,
        delivered_customer_timestamp,
        estimated_delivery_timestamp
    FROM silver_orders
    WHERE delivered_carrier_timestamp IS NULL
        AND status = 'delivered'
    ORDER BY 
        purchase_timestamp;
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,delivered_carrier_timestampTIMESTAMP,delivered_customer_timestampTIMESTAMP,estimated_delivery_timestampTIMESTAMP
2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,d77cf4be2654aa70ef150f8bfec076a6,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NULL,NULL,2017-06-23 00:00:00
2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,a2ac81ecc3704410ae240e74d4f0af40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NULL,2017-11-20 19:44:47,2017-11-14 00:00:00


> There are nearly 3000 customers who have made multiple orders, with the largest repeat customer making 17 different orders
* Consider counting overall products, or products per order for these customers

In [34]:
conn.query("""
    SELECT
        repeat_customer_id,
        COUNT(*) AS ct
    FROM silver_orders
    GROUP BY
        repeat_customer_id
    HAVING ct > 1
    ORDER BY 
        ct DESC;
""")

repeat_customer_idVARCHAR,ctBIGINT
8d50f5eadf50201ccdcedfb9e2ac8455,17
3e43e6105506432c953e165fb2acf44c,9
1b6c7548a2a1f9037c1fd3ddfed95f33,7
6469f99c1f9dfae7733b25662e7f1782,7
ca77025e7201e3b30c44b472ff346268,7
de34b16117594161a6a89c50b289d35a,6
f0e310a6839dce9de1638e0fe5ab282a,6
63cfc61cee11cbe306bff5857d00bfe4,6
dc813062e0fc23409cd255f7f53c7074,6
12f5d6e1cbf93dafd9dcc19095df0b3d,6


> Examples of orders that occured at the exact same time. Note the difference in carrier/customer delivery & estimated delivery
* Join _silver_customers_ to get geolocation to evaluate differences in shipping distance (also consider evaluating _freight_value_)

In [36]:
conn.query("""
    SELECT 
        purchase_timestamp,
        COUNT(*) AS ct
    FROM silver_orders
    GROUP BY 
        purchase_timestamp
    HAVING ct > 1
    ORDER BY 
        ct DESC;
""")

purchase_timestampTIMESTAMP,ctBIGINT
2017-11-20 11:46:30,3
2018-02-19 15:37:47,3
2018-08-02 12:06:07,3
2018-08-02 12:06:09,3
2018-08-02 12:05:26,3
2017-11-20 10:59:08,3
2018-06-01 13:39:44,3
2018-07-28 13:11:22,3
2018-04-11 10:48:14,3
2018-03-31 15:08:21,3


In [38]:
conn.query("""
    SELECT 
        order_id,
        customer_id,
        repeat_customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        delivered_carrier_timestamp,
        delivered_customer_timestamp,
        estimated_delivery_timestamp
    FROM silver_orders
    WHERE purchase_timestamp IN ('2017-11-20 11:46:30', '2018-02-19 15:37:47', '2018-08-02 12:06:07')
    ORDER BY 
        purchase_timestamp,
        repeat_customer_id;
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,delivered_carrier_timestampTIMESTAMP,delivered_customer_timestampTIMESTAMP,estimated_delivery_timestampTIMESTAMP
f9a0509d1e42522b51cc9789bbec8c1e,d842d0ebdf90c04a61d3c7ab3cab34a3,7f56220e6daf86b5071cbcc34d1fd022,delivered,2017-11-20 11:46:30,2017-11-20 12:08:31,2017-11-22 20:02:05,2017-11-28 19:56:38,2017-12-07 00:00:00
7bb854599c1f0c6bc0fc1dfd8a49e244,496092c6b78a42e0cd502fe42d698381,98eabb6f29d352e0468aac1372508400,delivered,2017-11-20 11:46:30,2017-11-20 12:07:01,2017-11-22 21:03:21,2017-12-19 15:58:55,2017-12-15 00:00:00
379663f7c5cf76e4a3cc7a2833071a39,398ddc24c8d53e2854a97f79965ba8f4,f89560e702ba272e6bc3b7477ead95c7,delivered,2017-11-20 11:46:30,2017-11-20 13:07:08,2017-11-22 12:31:54,2017-11-28 18:02:32,2017-12-01 00:00:00
62943af23e4392386dd77cdb5ed079e0,d7d084e643f5ec3bbb84f4e4639e510f,20ea57418f32eea0620ab3dc113f136d,delivered,2018-02-19 15:37:47,2018-02-19 16:46:57,2018-02-21 18:03:50,2018-03-10 17:12:17,2018-03-15 00:00:00
10e35bcb8797cb8e4cd3c28e01fca87a,341639b9cffe71e8c4a12cc5276ac193,ce1a8bd7a7c1eb40884dfd271cef3b3f,delivered,2018-02-19 15:37:47,2018-02-19 15:50:34,2018-02-20 21:37:51,2018-03-06 19:08:10,2018-03-14 00:00:00
e65296eb50fe1920930ee46ff3337238,320d65872c564439138f91907d31866c,dd404d81b12247da9794cf3fb53d36ba,shipped,2018-02-19 15:37:47,2018-02-19 16:35:29,2018-02-21 21:08:59,NULL,2018-03-14 00:00:00
91e573f4d0f3c29490414ff2f58a423c,cecbbfebf3d0f81403e8a660faa0e43b,6a7af8654e89e6156f9bb6d49daff205,delivered,2018-08-02 12:06:07,2018-08-02 13:04:19,2018-08-03 15:46:00,2018-08-07 17:16:10,2018-08-10 00:00:00
b2b8934746a9afa7b38c8c1f88268ecb,681986e7db7f47399570f6ed6a34f53c,9c3af16efacb7aa06aa3bc674556c5d6,delivered,2018-08-02 12:06:07,2018-08-02 13:04:06,2018-08-08 14:20:00,2018-08-13 21:38:56,2018-08-20 00:00:00
b2a3423735119b088bc62dd62cf053a8,7e6b8b2bdb547f7343c0d28f2ed5662e,ad85c133449f220e90bc455eac37c5e7,delivered,2018-08-02 12:06:07,2018-08-02 13:04:23,2018-08-10 12:27:00,2018-08-13 23:48:44,2018-08-09 00:00:00


> Approval wait time and distribution of wait time (by day) occurences

In [46]:
import matplotlib.pyplot as plt

df = conn.query("""
    SELECT 
        order_id,
        customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        approved_timestamp - purchase_timestamp AS approval_wait_time
    FROM silver_orders
    ORDER BY 
        approval_wait_time DESC;
""").to_df()

df.head()

,order_id,customer_id,status,purchase_timestamp,approved_timestamp,approval_wait_time
0,1612081119e8f23745698ad3367cc14b,20d32833d8983a835cafcd54099631a0,unavailable,2016-10-05 18:06:48,2017-04-11 15:17:38,187 days 21:10:50
1,2e5dc86c8c4aa663549caf5e31de840d,6225eed02b7d1a110b6e5b5dd4c8bd31,processing,2017-02-03 00:04:49,2017-04-04 10:56:48,60 days 10:51:59
2,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,32 days 16:02:44
3,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,32 days 13:01:41
4,0a5c74ccc786ced7903270de9d6c170a,c021456db05f8e71f0985bef8859b793,unavailable,2018-01-18 23:14:36,2018-02-20 12:05:54,32 days 12:51:18


In [71]:
type(df.approval_wait_time[0])
df['approval_wait_time_int'] = df.approval_wait_time.dt.days.astype('Int64')
# df[df['approval_wait_time_int'] > 5].plot.hist(bins = 50)
df.groupby('approval_wait_time_int')['approval_wait_time_int'].value_counts().reset_index()

,approval_wait_time_int,count
0,0,81861
1,1,12254
2,2,2904
3,3,1491
4,4,397
5,5,262
6,6,31
7,7,17
8,8,13
9,9,3


> All _orders_ time deltas

In [76]:
conn.query("""
    SELECT 
        order_id,
        customer_id,
        repeat_customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        approved_timestamp - purchase_timestamp AS purchase_to_approved_delta,
        delivered_carrier_timestamp,
        delivered_carrier_timestamp - approved_timestamp AS approval_to_carrier_delta,
        delivered_customer_timestamp,
        delivered_customer_timestamp - delivered_carrier_timestamp AS carrier_to_customer_delta,
        estimated_delivery_timestamp,
        estimated_delivery_timestamp - delivered_customer_timestamp AS delivered_vs_estimate_delta
    FROM silver_orders
    LIMIT 10;
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,purchase_to_approved_deltaINTERVAL,delivered_carrier_timestampTIMESTAMP,approval_to_carrier_deltaINTERVAL,delivered_customer_timestampTIMESTAMP,carrier_to_customer_deltaINTERVAL,estimated_delivery_timestampTIMESTAMP,delivered_vs_estimate_deltaINTERVAL
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,0:10:42,2017-10-04 19:55:00,"2 days, 8:47:45",2017-10-10 21:25:13,"6 days, 1:30:13",2017-10-18 00:00:00,"7 days, 2:34:47"
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,"1 day, 6:42:50",2018-07-26 14:31:00,11:06:33,2018-08-07 15:27:45,"12 days, 0:56:45",2018-08-13 00:00:00,"5 days, 8:32:15"
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,0:16:34,2018-08-08 13:50:00,4:54:37,2018-08-17 18:06:29,"9 days, 4:16:29",2018-09-04 00:00:00,"17 days, 5:53:31"
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,7c142cf63193a1473d2e66489a9ae977,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,0:17:53,2017-11-22 13:39:59,"3 days, 17:54:00",2017-12-02 00:28:42,"9 days, 10:48:43",2017-12-15 00:00:00,"12 days, 23:31:18"
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,1:01:50,2018-02-14 19:46:34,21:26:05,2018-02-16 18:17:02,"1 day, 22:30:28",2018-02-26 00:00:00,"9 days, 5:42:58"
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,0:13:08,2017-07-11 14:58:04,"1 day, 16:47:51",2017-07-26 10:57:55,"14 days, 19:59:51",2017-08-01 00:00:00,"5 days, 13:02:05"
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,36edbb3fb164b1f16485364b6fb04c73,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,"2 days, 1:03:09",NULL,NULL,NULL,NULL,2017-05-09 00:00:00,NULL
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,932afa1e708222e5821dac9cd5db4cae,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,0:11:41,2017-05-22 10:07:46,"5 days, 20:45:35",2017-05-26 12:55:51,"4 days, 2:48:05",2017-06-07 00:00:00,"11 days, 11:04:09"
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,39382392765b6dc74812866ee5ee92a7,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,"1 day, 8:21:38",2017-01-26 14:16:31,"1 day, 11:25:44",2017-02-02 14:08:10,"6 days, 23:51:39",2017-03-06 00:00:00,"31 days, 9:51:50"
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,299905e3934e9e181bfb2e164dd4b4f8,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,0:10:30,2017-08-10 19:45:24,"12 days, 7:39:52",2017-08-16 17:14:30,"5 days, 21:29:06",2017-08-23 00:00:00,"6 days, 6:45:30"


> Some orders have different _shipping_limit_timestamp_ values across products

In [80]:
conn.query("""
    SELECT *
    FROM silver_order_items
    WHERE order_id = 'b228e6b7a06468abcf8c203f8e86689c'
""")

order_idVARCHAR,product_idVARCHAR,item_sequenceBIGINT,shipping_limit_timestampTIMESTAMP,item_priceDOUBLE,shipping_costDOUBLE
b228e6b7a06468abcf8c203f8e86689c,31b8e7c2a22f3dba5183cc110bdd72f7,1,2018-08-14 16:10:10,199.6,19.49
b228e6b7a06468abcf8c203f8e86689c,53d71d8d9d6878f6f0b9fe0ed79445d5,2,2018-08-16 16:10:10,199.0,19.5


### Order SLA realted query

In [88]:
conn.query("""
    SELECT *
    FROM silver_customers
    LIMIT 5;
""")

customer_idVARCHAR,repeat_customer_idVARCHAR,zip_codeVARCHAR,cityVARCHAR,state_codeVARCHAR,state_nameVARCHAR,latitudeDOUBLE,longitudeDOUBLE
ce187ff9eaa89051f4c4fba7e99561f7,7cfc3a722d1fa04d8589096339b1698c,11675,caraguatatuba,SP,São Paulo,-23.6129560891208,-45.4240439493922
f8c3c173f3aebb9dc59a19669b25dddc,71b9eef1ec1e4ea4eb8f5db6307156f0,11680,ubatuba,SP,São Paulo,-23.43900930030584,-45.10246024854291
f41553f56ded7abddc895d3e4ad4c2c9,ac9af1cc78b26dc5d2e70333181833db,11677,caraguatatuba,SP,São Paulo,-23.623337437649386,-45.420671099371674
7fa99eee346e3a57243e5d30c76ad243,2ea0d4aef3699b453927c6e10886ba95,11668,caraguatatuba,SP,São Paulo,-23.71413379394921,-45.442959304558215
5447af43cbd7d4c3a958bbdd1327b6e3,8305c9033f18b2a9da785f80fbf80eb7,11630,ilhabela,SP,São Paulo,-23.82200708807961,-45.37499006419518


In [95]:
conn.query("""
    WITH latest_shipping_limit AS (
        SELECT
            order_id,
            MAX(shipping_limit_timestamp) AS latest_shipping_limit_timestamp
        FROM silver_order_items
        GROUP BY 
            order_id
    ), 
    sla_deltas AS (
        SELECT 
            o.order_id,
            customer_id,
            repeat_customer_id,
            status,
            purchase_timestamp,
            approved_timestamp,
            approved_timestamp - purchase_timestamp AS purchase_to_approved_delta,
            delivered_carrier_timestamp,
            delivered_carrier_timestamp - approved_timestamp AS approval_to_carrier_delta,
            oi.latest_shipping_limit_timestamp,
            oi.latest_shipping_limit_timestamp  - delivered_carrier_timestamp AS carrier_vs_shipping_limit_delta,
            delivered_customer_timestamp,
            delivered_customer_timestamp - delivered_carrier_timestamp AS carrier_to_customer_delta,
            estimated_delivery_timestamp,
            estimated_delivery_timestamp - delivered_customer_timestamp AS delivered_vs_estimate_delta
        FROM silver_orders o
        LEFT JOIN latest_shipping_limit oi 
            ON o.order_id = oi.order_id
    )
    SELECT COUNT(*) FROM sla_deltas;
""")

count_star()BIGINT
99441


In [97]:
conn.query("""
    SELECT COUNT(*) FROM silver_customers;
""")

count_star()BIGINT
15083733


In [104]:
conn.query("""
    WITH geolocation_agg AS (
        SELECT 
            geolocation_zip_code_prefix AS zip_code,
            AVG(geolocation_lat) AS avg_latitude,
            AVG(geolocation_lng) AS avg_longitude
        FROM bronze_geolocation
        GROUP BY
            geolocation_zip_code_prefix
    )
    SELECT
        customer_id,
        customer_unique_id AS repeat_customer_id,
        customer_zip_code_prefix AS zip_code,
        customer_city AS city,
        customer_state AS state_code,
        bs.state_name,
        geolocation_lat AS latitude,
        geolocation_lng AS longitude
    FROM bronze_customers c
    INNER JOIN geolocation_agg g 
        ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
    LEFT JOIN bronze_brazil_state_code_ref bs 
        ON c.customer_state = bs.state_code
""")

geolocation_zip_code_prefixVARCHAR,geolocation_latDOUBLE,geolocation_lngDOUBLE,geolocation_cityVARCHAR,geolocation_stateVARCHAR
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP
01012,-23.547762303364266,-46.63536053788448,são paulo,SP
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [103]:
conn.query("""
    WITH geolocation_agg AS (
        SELECT 
            *
        FROM bronze_geolocation
    )
        SELECT
            customer_id,
            customer_unique_id AS repeat_customer_id,
            customer_zip_code_prefix AS zip_code,
            customer_city AS city,
            customer_state AS state_code,
            bs.state_name,
            geolocation_lat AS latitude,
            geolocation_lng AS longitude
        FROM bronze_customers c
        INNER JOIN bronze_geolocation g 
            ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
        LEFT JOIN bronze_brazil_state_code_ref bs 
            ON c.customer_state = bs.state_code
    )
    SELECT *
    FROM tmp
    WHERE customer_id = 'fba9938946afd9cd58e7be16ddbe9c19'
""")

customer_idVARCHAR,repeat_customer_idVARCHAR,zip_codeVARCHAR,cityVARCHAR,state_codeVARCHAR,state_nameVARCHAR,latitudeDOUBLE,longitudeDOUBLE
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.905816548704518,-43.106988856771984
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.902305918691127,-43.11254462280002
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.904566594523498,-43.11049125088019
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.902574860407324,-43.109192038980986
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.90749962932978,-43.10616955390452
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.90431897584976,-43.113001263950586
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.899013761065042,-43.106746265047086
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.90740120734021,-43.09858491006412
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.90306128169552,-43.109013904091526
fba9938946afd9cd58e7be16ddbe9c19,0ca97900e34e5815ebf8b4e7f7883e74,24220,niteroi,RJ,Rio de Janeiro,-22.901422106036033,-43.11080235027977


In [16]:
conn.query("""
    SELECT *
    FROM gold_customers
    LIMIT 10;
""")

customer_idVARCHAR,repeat_customer_idVARCHAR,zip_codeVARCHAR,cityVARCHAR,state_codeVARCHAR,state_nameVARCHAR,avg_latitudeDOUBLE,avg_longitudeDOUBLE
6775882273c160e7c61df402719d4ca6,213dc56df4763e3150c5972390f9b2e4,05465,sao paulo,SP,São Paulo,-23.554149135449844,-46.71601739435356
21fa606c2b3d80fa8910fa958df41cb5,a78da8d7418aa0b0fd878582e1ce84bf,05436,sao paulo,SP,São Paulo,-23.554461114753504,-46.68535387537024
3e21de36f45bda5fc08a8ddf5ff1ea3f,6770c2cc4bb1ee7cb6fad85b5acf4a2a,05416,sao paulo,SP,São Paulo,-23.56094857667274,-46.68842233187829
355670f34587eb802a40006a0fe3374f,e57884a347f585cce8f8b6fb08ba39c2,05404,sao paulo,SP,São Paulo,-23.56383812827326,-46.68371871693125
d60157b6edc4c62efe9bda6b47cadf5d,c2f1bd7529dcb6abfa92705130fe3299,05567,sao paulo,SP,São Paulo,-23.594027301222617,-46.786173435750975
ed1c6c1cc4e0fff91a4f76e79670b188,bbb24b0ebbe51990016de5086071f757,05588,sao paulo,SP,São Paulo,-23.57601657708377,-46.74118636634989
ee3dd978021aef55ba8a583afd608049,cc15a06fcaf30c26c11f641c5d753080,05545,sao paulo,SP,São Paulo,-23.58665318740992,-46.76690112639909
0ad37f50dbe78d9c1292f4ab3217dea5,9577832effc61fd2df9e05cac75f04de,05586,sao paulo,SP,São Paulo,-23.57072227332635,-46.72857079879624
48cabebc3016b12079f4d41904c645b3,7bf1c59401206e31a78523077f3a1144,05521,sao paulo,SP,São Paulo,-23.58865488968895,-46.72665430092211
cab4cad0f7657576558c907ad8e307fd,0dc9c8d77cfdc1d53d8378286c77de05,05585,sao paulo,SP,São Paulo,-23.57111603341937,-46.725326083567445


In [ ]:
conn.query("""
    WITH latest_shipping_limit AS (
        SELECT
            order_id,
            MAX(shipping_limit_timestamp) AS latest_shipping_limit_timestamp
        FROM silver_order_items
        GROUP BY 
            order_id
    ),
    sla_deltas AS (
        SELECT 
            o.order_id,
            o.customer_id,
            o.repeat_customer_id,
            c.zip_code,
            c.city,
            c.state_name AS state,
            c.avg_latitude,
            c.avg_longitude,
            status,
            purchase_timestamp,
            approved_timestamp,
            approved_timestamp - purchase_timestamp AS purchase_to_approved_delta,
            delivered_carrier_timestamp,
            delivered_carrier_timestamp - approved_timestamp AS approval_to_carrier_delta,
            oi.latest_shipping_limit_timestamp,
            oi.latest_shipping_limit_timestamp  - delivered_carrier_timestamp AS carrier_vs_shipping_limit_delta,
            delivered_customer_timestamp,
            delivered_customer_timestamp - delivered_carrier_timestamp AS carrier_to_customer_delta,
            estimated_delivery_timestamp,
            estimated_delivery_timestamp - delivered_customer_timestamp AS delivered_vs_estimate_delta,
            CASE 
                WHEN delivered_customer_timestamp > estimated_delivery_timestamp THEN 'Late'
                WHEN 
        FROM silver_orders o
        LEFT JOIN latest_shipping_limit oi 
            ON o.order_id = oi.order_id
        LEFT JOIN gold_customers c 
            ON o.customer_id = c.customer_id
    )
    SELECT *
    FROM sla_deltas
    ORDER BY 
        order_id;
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,zip_codeVARCHAR,cityVARCHAR,stateVARCHAR,avg_latitudeDOUBLE,avg_longitudeDOUBLE,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,purchase_to_approved_deltaINTERVAL,delivered_carrier_timestampTIMESTAMP,approval_to_carrier_deltaINTERVAL,latest_shipping_limit_timestampTIMESTAMP,carrier_vs_shipping_limit_deltaINTERVAL,delivered_customer_timestampTIMESTAMP,carrier_to_customer_deltaINTERVAL,estimated_delivery_timestampTIMESTAMP,delivered_vs_estimate_deltaINTERVAL
00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,Rio de Janeiro,-21.762775347060224,-41.30963269524094,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,0:46:33,2017-09-19 18:34:16,"6 days, 8:48:41",2017-09-19 09:45:35,"-1 day, 15:11:19",2017-09-20 23:43:48,"1 day, 5:09:32",2017-09-29 00:00:00,"8 days, 0:16:12"
00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,São Paulo,-20.22052678746204,-50.90342436588621,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,0:12:07,2017-05-04 14:35:00,"8 days, 3:29:47",2017-05-03 11:05:13,"-2 days, 20:30:13",2017-05-12 16:04:24,"8 days, 1:29:24",2017-05-15 00:00:00,"2 days, 7:55:36"
000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,Minas Gerais,-19.870304664180164,-44.59332570290271,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,0:14:59,2018-01-16 12:36:48,"1 day, 21:48:18",2018-01-18 14:48:30,"2 days, 2:11:42",2018-01-22 13:19:16,"6 days, 0:42:28",2018-02-05 00:00:00,"13 days, 10:40:44"
00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,São Paulo,-23.08992524585618,-46.611654225087754,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,0:09:43,2018-08-10 13:28:00,"2 days, 3:17:42",2018-08-15 10:10:18,"4 days, 20:42:18",2018-08-14 13:32:39,"4 days, 0:04:39",2018-08-20 00:00:00,"5 days, 10:27:21"
00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,São Paulo,-23.243401530196394,-46.827614126118235,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,0:12:22,2017-02-16 09:46:09,"11 days, 19:35:56",2017-02-13 13:57:51,"-3 days, 4:11:42",2017-03-01 16:42:31,"13 days, 6:56:22",2017-03-17 00:00:00,"15 days, 7:17:29"
00048cc3ae777c65dbb7d2a0634bc1ea,816cbea969fe5b689b39cfc97a506742,85c835d128beae5b4ce8602c491bf385,38017,uberaba,Minas Gerais,-19.740198535741925,-47.92479298786426,delivered,2017-05-15 21:42:34,2017-05-17 03:55:27,"1 day, 6:12:53",2017-05-17 11:05:55,7:10:28,2017-05-23 03:55:27,"5 days, 16:49:32",2017-05-22 13:44:35,"5 days, 2:38:40",2017-06-06 00:00:00,"14 days, 10:15:25"
00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,635d9ac1680f03288e72ada3a1035803,16700,guararapes,São Paulo,-21.25058666062858,-50.646445039254175,delivered,2017-12-10 11:53:48,2017-12-10 12:10:31,0:16:43,2017-12-12 01:07:48,"1 day, 12:57:17",2017-12-14 12:10:31,"2 days, 11:02:43",2017-12-18 22:03:38,"6 days, 20:55:50",2018-01-04 00:00:00,"16 days, 1:56:22"
000576fe39319847cbb9d288c5617fa6,9ed5e522dd9dd85b4af4a077526d8117,fda4476abb6307ab3c415b7e6d026526,11702,praia grande,São Paulo,-24.01439467431421,-46.438358319817674,delivered,2018-07-04 12:08:27,2018-07-05 16:35:48,"1 day, 4:27:21",2018-07-05 12:15:00,"-1 day, 19:39:12",2018-07-10 12:30:45,"5 days, 0:15:45",2018-07-09 14:04:07,"4 days, 1:49:07",2018-07-25 00:00:00,"15 days, 9:55:53"
0005a1a1728c9d785b8e2b08b904576c,16150771dfd4776261284213b89c304e,639d23421f5517f69d0c3d6e6564cf0e,11075,santos,São Paulo,-23.95460047631286,-46.3377311534732,delivered,2018-03-19 18:40:33,2018-03-20 18:35:21,23:54:48,2018-03-28 00:37:42,"7 days, 6:02:21",2018-03-26 18:31:29,"-2 days, 17:53:47",2018-03-29 18:17:31,"1 day, 17:39:49",2018-03-29 00:00:00,"-1 day, 5:42:29"
0005f50442cb953dcd1d21e1fb

In [14]:
conn.query("""
    SELECT 
        order_id,
        customer_id,
        repeat_customer_id,
        status,
        purchase_timestamp,
        approved_timestamp,
        delivered_carrier_timestamp,
        delivered_customer_timestamp,
        estimated_delivery_timestamp
    FROM silver_orders
    LIMIT 10;
""")

order_idVARCHAR,customer_idVARCHAR,repeat_customer_idVARCHAR,statusVARCHAR,purchase_timestampTIMESTAMP,approved_timestampTIMESTAMP,delivered_carrier_timestampTIMESTAMP,delivered_customer_timestampTIMESTAMP,estimated_delivery_timestampTIMESTAMP
8999cbe9107b9b5f08d0901aa10dbb67,46afa5018cb6e3d121c49704ba4bd938,4fcf005606ebb2a031f3f9b5e22fa90b,delivered,2018-05-16 17:06:00,2018-05-18 04:36:10,2018-05-18 13:23:00,2018-06-06 20:12:53,2018-06-07 00:00:00
f8898796f33b7a0741dc0cf4f54c0001,f04bc3f68cb2a1fff4fb642b10c6ecdb,91f0db1a51449898b8f41e565fcffaca,delivered,2017-10-03 07:57:54,2017-10-03 08:07:19,2017-10-03 16:32:58,2017-10-06 17:50:26,2017-10-26 00:00:00
3b9119591008704806aad33e4ebd3201,6e441d93f95beaaf3a87e6982439120d,d8eafc07c9e450d0bddbf75e0c6562ae,delivered,2017-11-25 12:47:41,2017-11-25 12:57:31,2017-11-29 22:58:34,2017-12-14 00:17:50,2017-12-20 00:00:00
75394bdb62cb6061c075d73965d65bc7,48d52656f1b2e51c6316984afd1f50ac,e7e6adc70176b14f21be09c6346bc853,delivered,2018-02-26 20:28:48,2018-02-26 21:21:08,2018-03-01 22:23:44,2018-03-20 19:18:42,2018-03-23 00:00:00
daa13cb5f8efbd2ee69043963b0f233c,6a5b19b0670d2e6bfe0ac7b4c858f58f,6554c017c778ada342413c99dac69a3f,delivered,2017-05-02 20:37:35,2017-05-03 15:15:22,2017-05-05 14:47:53,2017-05-17 14:06:59,2017-05-25 00:00:00
99dd7eacf34dfd90a3277bdcdfb454ac,ad7e368f155a2a9da1d1599cb84827d0,767afad3c097cfd4cc95ada85d1dc453,delivered,2017-05-12 13:36:45,2017-05-13 03:05:23,2017-05-15 10:47:48,2017-05-22 14:33:47,2017-05-30 00:00:00
a3629884bf8956b0f572069b8a20cc55,6b04ea93ce93cef2a5fbdfe302e21331,c73db85789b600b05c503d21441940f9,delivered,2017-10-03 13:26:21,2017-10-03 13:49:25,2017-10-04 11:12:01,2017-10-14 00:27:42,2017-11-17 00:00:00
6ef662a8188539a6881080ef916fddf0,0f377e7c0d721d5644a35bb1665e6499,fdcca758610ea813796898f0169bf453,delivered,2018-05-15 16:39:20,2018-05-16 16:39:02,2018-05-17 15:28:00,2018-06-04 12:51:51,2018-05-30 00:00:00
730c83933acab283b21bae7b3fcc91d1,6e37091e698b4eededd017fa87a1ce5b,d1b1a515d00c439c119352339757376c,delivered,2018-07-05 08:54:50,2018-07-05 16:07:03,2018-07-05 14:24:00,2018-07-11 16:16:48,2018-08-03 00:00:00
1915bc319651dfbfa6563a6d19d558c5,093cc8dd94c7aae7b33742a5ff3be232,b2721cd583ed718f04e636e3cbbaabc0,delivered,2017-05-18 14:15:54,2017-05-18 14:31:21,2017-05-22 06:11:10,2017-05-24 15:53:06,2017-05-31 00:00:00


In [9]:
conn.query(""" 
    SELECT 
        oi.order_id,
        oi.product_id,
        o.purchase_timestamp,
        pr.category,
        oi.item_sequence,
        o.status,
        oi.item_price,
        oi.shipping_cost,
        p.payment_sequence,
        p.type,
        p.installments,
        p.value
    FROM silver_order_items oi
    LEFT JOIN silver_orders o
        ON oi.order_id = o.order_id
    LEFT JOIN silver_payments p
        ON oi.order_id = p.order_id
    LEFT JOIN silver_products pr
        ON oi.product_id = pr.product_id
    WHERE oi.order_id = 'ca3625898fbd48669d50701aba51cd5f'
""")

order_idVARCHAR,product_idVARCHAR,purchase_timestampTIMESTAMP,categoryVARCHAR,item_sequenceBIGINT,statusVARCHAR,item_priceDOUBLE,shipping_costDOUBLE,payment_sequenceBIGINT,typeVARCHAR,installmentsBIGINT,valueDOUBLE
ca3625898fbd48669d50701aba51cd5f,1065e0ebef073787a7bf691924c60eeb,2018-08-12 02:11:20,construction tools construction,1,delivered,309.0,1.84,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,0cf2faf9749f53924cea652a09d8e327,2018-08-12 02:11:20,construction tools construction,2,delivered,33.9,1.84,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,0de59eddc63167215c972b0d785ffa7b,2018-08-12 02:11:20,construction tools construction,3,delivered,159.0,3.67,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,0de59eddc63167215c972b0d785ffa7b,2018-08-12 02:11:20,construction tools construction,4,delivered,159.0,3.67,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,5dae498eff2d80057f56122235a36aff,2018-08-12 02:11:20,construction tools construction,5,delivered,95.9,0.15,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,4a5c3967bfd3629fe07ef4d0cc8c3818,2018-08-12 02:11:20,construction tools construction,6,delivered,109.9,0.15,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,678c229b41c0e497d35a25a8be1cc631,2018-08-12 02:11:20,construction tools construction,7,delivered,95.9,0.15,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,21b524c4c060169fa75ccf08c7da4627,2018-08-12 02:11:20,construction tools construction,8,delivered,63.7,0.15,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,309dd69eb83cea38c51709d62befe1a4,2018-08-12 02:11:20,construction tools construction,9,delivered,56.0,3.68,1,credit_card,10,1157.28
ca3625898fbd48669d50701aba51cd5f,309dd69eb83cea38c51709d62befe1a4,2018-08-12 02:11:20,construction tools construction,10,delivered,56.0,3.68,1,credit_card,10,1157.28


### Testing UDF at view call level

In [9]:
from deep_translator import MyMemoryTranslator
from duckdb.sqltypes import VARCHAR

# User-defined functions (UDF) definitions 
def translate_to_english (text: str) -> str:
    en_text = MyMemoryTranslator(source = 'pt-PT', target = 'en-US').translate(text)
    return en_text

# Register UDF in duckdb
try:
    conn.remove_function('translate_to_english')
except Exception:
    pass

conn.create_function('translate_to_english', translate_to_english, [VARCHAR], VARCHAR)

In [17]:
conn.close()